In [1]:
library(tidyverse)
library(data.table)
library(purrr)

-- Attaching core tidyverse packages ---------------------------------------------------------------- tidyverse 2.0.0 --
v dplyr     1.1.2     v readr     2.1.4
v forcats   1.0.0     v stringr   1.5.0
v ggplot2   3.4.3     v tibble    3.2.1
v lubridate 1.9.2     v tidyr     1.3.0
v purrr     1.0.1     
-- Conflicts ---------------------------------------------------------------------------------- tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: 'data.table'


The following objects are masked from 'package:lubridate':

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from 'package:dplyr':

    between, first, last


The following object is masked from 'package:purrr':

    transpose




In [2]:
tab <- fread("../no prunning/no_pruning.csv")
tab <- tab[,14:(ncol(tab) - 1)]

In [107]:
col_list  <- as.list(colnames(tab)) %>% 
    map(., ~str_replace(.x, ".*_", ""))
length(col_list)

[1] 302

In [4]:
for (rs in col_list) {
   cat(rs, '\n')
}

#the below list is copied and pasted into VEP web interface

rs4713859 
rs3736228 
rs10476909 
rs1379544 
rs921978 
rs447950 
rs414582 
rs3733658 
rs2400891 
rs10062536 
rs4988300 
rs4713858 
rs1363629 
rs6861564 
rs4713854 
rs7728256 
rs452366 
rs474742 
rs476741 
rs241280 
rs2454104 
rs3823434 
rs312009 
rs314750 
rs3781586 
rs3781590 
rs634008 
rs4409073 
rs6887452 
rs353237 
rs10061434 
rs599083 
rs12417014 
rs12659504 
rs2306862 
rs7936582 
rs17724303 
rs11826287 
rs2267665 
rs1883322 
rs632605 
rs116213531 
rs75210709 
rs2508835 
rs72823547 
rs11228270 
rs353294 
rs115128769 
rs11574424 
rs660925 
rs115658620 
rs77381812 
rs2125130 
rs72936563 
rs2895862 
rs353248 
rs6861618 
rs78578633 
rs74724919 
rs62378023 
rs118098503 
rs353241 
rs111416523 
rs4988332 
rs35388122 
rs113800014 
rs6894391 
rs72932379 
rs117311236 
rs35928487 
rs114688286 
rs62378010 
rs6889482 
rs4988321 
rs118062233 
rs77489920 
rs13214733 
rs72823546 
rs10454990 
rs76225083 
rs75177427 
rs7342161 
rs74474766 
rs11228184 
rs75148296 
rs115413638 
rs57928864 
rs74697984

#### Function to clean marker names

In [85]:
clean_marker <- function(marker) {
  marker <- tolower(marker)
  marker <- gsub("gsa-|seq-", "", marker)
  marker <- dplyr::recode(marker, "ilmnseq_6:35378798" = "rs9658134")
  return(marker)
}

In [114]:
snp <- fread("../feature_tsv/ancestry_allele_frequency/plink.assoc.logistic.tsv") %>%
  filter(TEST == "ADD") %>%   
  mutate(SNP = clean_marker(SNP)) %>% 
  select(c(SNP, ORX, A1, P )) %>% 
  rename(allele = A1, marker = SNP, OR = ORX) %>% 
  filter(marker %in% col_list)  

In [123]:
vep <- fread("./vep_output/vep_output.txt", sep="\t")

In [121]:
setdiff(col_list, snp$marker )

[[1]]
[1] "rs72932379"

[[2]]
[1] "rs72823546"

[[3]]
[1] "rs75177427"

[[4]]
[1] "rs7761870"

[[5]]
[1] "rs6457821"

[[6]]
[1] "rs111776064"

[[7]]
[1] "rs3823433"

[[8]]
[1] "rs6457813"

[[9]]
[1] "rs9658134"

In [124]:
vep <- vep %>% 
    rename(marker = `#Uploaded_variation`, allele = Allele, consequence = Consequence , impact = IMPACT, symbol = SYMBOL, biotype = BIOTYPE, `CADD_phred` = `CADD_PHRED`, Canonical = CANONICAL) #%>% 
#     select(c(marker, allele, symbol, consequence, impact, biotype, Feature_type, Feature, `CADD_phred`, Canonical, AF, AFR_AF, AMR_AF, EAS_AF, EUR_AF, SAS_AF) )

In [135]:
combined <- vep %>% 
left_join(snp, by= c("marker", "allele")) %>% 
arrange(P) %>% 
mutate(P = round(P, 3)) %>% 
distinct()

combined$EXON <- as.character(combined$EXON)
combined$INTRON <- as.character(combined$INTRON)

In [136]:
combined

marker,Location,allele,consequence,impact,symbol,Gene,Feature_type,Feature,biotype,...,TRANSCRIPTION_FACTORS,GO,PHENOTYPES,Mastermind_MMID3,Geno2MP_HPO_count,Geno2MP_URL,CADD_phred,CADD_RAW,OR,P
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,...,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
rs12731961,1:230923313-230923313,T,missense_variant,MODERATE,CAPN9,ENSG00000135773,Transcript,ENST00000271971.2,protein_coding,...,-,"GO:0004198:calcium-dependent_cysteine-type_endopeptidase_activity,GO:0005509:calcium_ion_binding,GO:0005575:cellular_component,GO:0005622:intracellular,GO:0006508:proteolysis,GO:0007586:digestion",-,"CAPN9:R459W,CAPN9:R496W,CAPN9:R522W",-,-,22.1,2.314699,2.6060,0.000
rs12731961,1:230923313-230923313,T,missense_variant,MODERATE,CAPN9,ENSG00000135773,Transcript,ENST00000354537.1,protein_coding,...,-,"GO:0004198:calcium-dependent_cysteine-type_endopeptidase_activity,GO:0005509:calcium_ion_binding,GO:0005575:cellular_component,GO:0005622:intracellular,GO:0006508:proteolysis,GO:0007586:digestion",-,"CAPN9:R459W,CAPN9:R496W,CAPN9:R522W",-,-,22.1,2.314699,2.6060,0.000
rs12731961,1:230923313-230923313,T,missense_variant,MODERATE,CAPN9,ENSG00000135773,Transcript,ENST00000366666.2,protein_coding,...,-,"GO:0004198:calcium-dependent_cysteine-type_endopeptidase_activity,GO:0005509:calcium_ion_binding,GO:0005622:intracellular,GO:0006508:proteolysis",-,"CAPN9:R459W,CAPN9:R496W,CAPN9:R522W",-,-,22.1,2.314699,2.6060,0.000
rs12731961,1:230923313-230923313,T,"intron_variant,non_coding_transcript_variant",MODIFIER,RP11-99J16__A.2,ENSG00000244137,Transcript,ENST00000412344.1,antisense,...,-,-,-,-,-,-,22.1,2.314699,2.6060,0.000
rs12731961,1:230923313-230923313,T,"intron_variant,non_coding_transcript_variant",MODIFIER,RP11-99J16__A.2,ENSG00000244137,Transcript,ENST00000428480.1,antisense,...,-,-,-,-,-,-,22.1,2.314699,2.6060,0.000
rs12731961,1:230923313-230923313,T,"intron_variant,non_coding_transcript_variant",MODIFIER,RP11-99J16__A.2,ENSG00000244137,Transcript,ENST00000452640.1,antisense,...,-,-,-,-,-,-,22.1,2.314699,2.6060,0.000
rs12731961,1:230923313-230923313,T,upstream_gene_variant,MODIFIER,CAPN9,ENSG00000135773,Transcript,ENST00000480004.1,processed_transcript,...,-,-,-,-,-,-,22.1,2.314699,2.6060,0.000
rs12082061,1:230924905-230924905,T,intron_variant,MODIFIER,CAPN9,ENSG00000135773,Transcript,ENST00000271971.2,protein_coding,...,-,"GO:0004198:calcium-dependent_cysteine-type_endopeptidase_activity,GO:0005509:calcium_ion_binding,GO:0005575:cellular_component,GO:0005622:intracellular,GO:0006508:proteolysis,GO:0007586:digestion",-,"CAPN9:E470int,CAPN9:E470inta,CAPN9:E507int,CAPN9:E507inta,CAPN9:E533int,CAPN9:E533inta",-,-,0.271,-0.343595,2.1740,0.000
rs12082061,1:230924905-230924905,T,intron_variant,MODIFIER,CAPN9,ENSG00000135773,Transcript,ENST00000354537.1,protein_coding,...,-,"GO:0004198:calcium-dependent_cysteine-type_endopeptidase_activity,GO:0005509:calcium_ion_binding,GO:0005575:cellular_component,GO:0005622:intracellular,GO:0006508:proteolysis,GO:0007586:digestion",-,"CAPN9:E470int,CAPN9:E470inta,CAPN9:E507int,CAPN9:E507inta,CAPN9:E533int,CAPN9:E533inta",-,-,0.271,-0.343595,2.1740,0.000


In [138]:
rank <- fread("./xgboost_output_feature_importance")  %>% 
    rename(variant = V1, xgboost_score = V2) %>% 
    mutate(variant = clean_marker(variant)) %>% 
    filter(grepl("rs", variant)) %>% 
    separate(variant, into = c("gene", "marker"), sep = "_")  %>% 
    arrange(desc(xgboost_score)) %>% 
    filter(xgboost_score != 0 )



In [148]:
withUnpruned <- left_join(combined, rank, by="marker") %>% 
distinct()

In [149]:
withUnpruned

marker,Location,allele,consequence,impact,symbol,Gene,Feature_type,Feature,biotype,...,PHENOTYPES,Mastermind_MMID3,Geno2MP_HPO_count,Geno2MP_URL,CADD_phred,CADD_RAW,OR,P,gene,xgboost_score
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,...,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
rs12731961,1:230923313-230923313,T,missense_variant,MODERATE,CAPN9,ENSG00000135773,Transcript,ENST00000271971.2,protein_coding,...,-,"CAPN9:R459W,CAPN9:R496W,CAPN9:R522W",-,-,22.1,2.314699,2.6060,0.000,NA,NA
rs12731961,1:230923313-230923313,T,missense_variant,MODERATE,CAPN9,ENSG00000135773,Transcript,ENST00000354537.1,protein_coding,...,-,"CAPN9:R459W,CAPN9:R496W,CAPN9:R522W",-,-,22.1,2.314699,2.6060,0.000,NA,NA
rs12731961,1:230923313-230923313,T,missense_variant,MODERATE,CAPN9,ENSG00000135773,Transcript,ENST00000366666.2,protein_coding,...,-,"CAPN9:R459W,CAPN9:R496W,CAPN9:R522W",-,-,22.1,2.314699,2.6060,0.000,NA,NA
rs12731961,1:230923313-230923313,T,"intron_variant,non_coding_transcript_variant",MODIFIER,RP11-99J16__A.2,ENSG00000244137,Transcript,ENST00000412344.1,antisense,...,-,-,-,-,22.1,2.314699,2.6060,0.000,NA,NA
rs12731961,1:230923313-230923313,T,"intron_variant,non_coding_transcript_variant",MODIFIER,RP11-99J16__A.2,ENSG00000244137,Transcript,ENST00000428480.1,antisense,...,-,-,-,-,22.1,2.314699,2.6060,0.000,NA,NA
rs12731961,1:230923313-230923313,T,"intron_variant,non_coding_transcript_variant",MODIFIER,RP11-99J16__A.2,ENSG00000244137,Transcript,ENST00000452640.1,antisense,...,-,-,-,-,22.1,2.314699,2.6060,0.000,NA,NA
rs12731961,1:230923313-230923313,T,upstream_gene_variant,MODIFIER,CAPN9,ENSG00000135773,Transcript,ENST00000480004.1,processed_transcript,...,-,-,-,-,22.1,2.314699,2.6060,0.000,NA,NA
rs12082061,1:230924905-230924905,T,intron_variant,MODIFIER,CAPN9,ENSG00000135773,Transcript,ENST00000271971.2,protein_coding,...,-,"CAPN9:E470int,CAPN9:E470inta,CAPN9:E507int,CAPN9:E507inta,CAPN9:E533int,CAPN9:E533inta",-,-,0.271,-0.343595,2.1740,0.000,agt,51.47215
rs12082061,1:230924905-230924905,T,intron_variant,MODIFIER,CAPN9,ENSG00000135773,Transcript,ENST00000354537.1,protein_coding,...,-,"CAPN9:E470int,CAPN9:E470inta,CAPN9:E507int,CAPN9:E507inta,CAPN9:E533int,CAPN9:E533inta",-,-,0.271,-0.343595,2.1740,0.000,agt,51.47215


In [131]:
combined <- inner_join(combined, rank) 

Joining with `by = join_by(marker)`


In [142]:
setdiff(rank$marker, combined$marker)

character(0)

In [143]:
xslx_format <- function(df, col) {
    df[[col]] <- as.character(df[[col]])
    df[[col]] <- gsub(",", ";", df[[col]])
    df[[col]]<- gsub("/","of", df[[col]])
    return(df)
}

In [144]:
xslx_format <- function(col) {
  col <- gsub(",", ";", col)
  col <- gsub("/", "of", col)
  return(col)
}

In [150]:
withUnpruned <- withUnpruned %>% 
    mutate(across(where(is.character), xslx_format)) %>% 
      distinct()

In [67]:
combined <- combined %>%
  mutate(across(where(is.character), xslx_format)) %>% 
  distinct()

In [69]:
write.csv(combined , "./result/pruned_vep.csv", quote=FALSE, row.names=FALSE)

In [152]:
write.csv(withUnpruned, "./result/nonpruned_vep.csv")

In [233]:
GTEX <- combined %>% 
distinct(marker, allele, xgboost_score) %>% 
arrange(desc(xgboost_score))
GTEX

marker,allele,xgboost_score
<chr>,<chr>,<dbl>
rs6887452,G,100.00000
rs56022120,T,75.66017
rs9658119,C,73.18871
rs17879591,T,69.74603
rs12462974,C,64.67756
rs79722771,T,64.26292
rs6799185,T,60.02445
rs12082061,T,51.47215
rs11228184,A,50.88202


In [234]:
write.csv(GTEX , "./result/gtex.csv", quote=FALSE, row.names=FALSE)